In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print("Kích thước tập train:", train_df.shape)
print("Kích thước tập test:", test_df.shape)
train_df.head()

Kích thước tập train: (8693, 14)
Kích thước tập test: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True



Bước đầu tiên là tải thư viện, nạp dữ liệu và kiểm tra tổng quan các đặc trưng (features) để tìm ra phương án xử lý các giá trị bị thiếu (missing values).

In [10]:
def preprocess_data(df):
    df = df.copy()
    
    df['Cabin'] = df['Cabin'].fillna('Z/9999/Z') 
    df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['Num'] = df['Num'].astype(int)
    
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spend_cols:
        df[col] = df[col].fillna(0)
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['VIP'] = df['VIP'].fillna(False)
    df['CryoSleep'] = df['CryoSleep'].fillna(False)
    df['HomePlanet'] = df['HomePlanet'].fillna(df['HomePlanet'].mode()[0])
    df['Destination'] = df['Destination'].fillna(df['Destination'].mode()[0])
    
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
    group_counts = df['Group'].value_counts()
    df['GroupSize'] = df['Group'].map(group_counts)
    
    df = df.drop(['PassengerId', 'Name', 'Cabin', 'Group'], axis=1)
    
    return df

X_train_full = preprocess_data(train_df)
X_test_final = preprocess_data(test_df)

y = X_train_full['Transported'].astype(int)
X = X_train_full.drop('Transported', axis=1)

cat_features = ['HomePlanet', 'Destination', 'CryoSleep', 'VIP', 'Deck', 'Side']
le = LabelEncoder()

for col in cat_features:
    X[col] = le.fit_transform(X[col].astype(str))
    X_test_final[col] = le.transform(X_test_final[col].astype(str))

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


Dữ liệu thô chứa rất nhiều giá trị null và các cột văn bản (text) không thể trực tiếp đưa vào mô hình. Kỹ thuật Feature Engineering ở đây bao gồm:
*   **Tách cột `Cabin`**: Cấu trúc của Cabin là `Deck/Num/Side`. Việc tách đặc trưng này ra làm 3 cột riêng biệt (`Deck`, `Num`, `Side`) mang lại rất nhiều thông tin quan trọng.
*   **Tổng chi tiêu (`TotalSpend`)**: Hành khách ngủ đông (`CryoSleep = True`) sẽ không tiêu tiền. Ta tổng hợp các khoản chi tiêu (RoomService, FoodCourt, ShoppingMall, Spa, VRDeck) lại thành một cột mới.
*   **Xử lý Missing Values**: Điền các giá trị phân loại (Categorical) bằng giá trị xuất hiện nhiều nhất (Mode) và các giá trị số (Numerical) bằng trung vị (Median).
*   **Label Encoding**: Chuyển đổi các cột dạng chuỗi (String) sang số nguyên (Integer) để các mô hình học máy có thể hiểu được.

In [11]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=42, eval_metric='logloss')
lgbm_model = LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42)
cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, random_state=42, verbose=0)

voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('cat', cat_model)
    ],
    voting='soft'
)

models = {
    'Bagging (Random Forest)': rf_model,
    'Boosting (XGBoost)': xgb_model,
    'Boosting (LightGBM)': lgbm_model,
    'Boosting (CatBoost)': cat_model,
    'Ensemble (Voting)': voting_clf
}

print("BENCHMARK TRÊN TẬP VALIDATION (20% DỮ LIỆU):")
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    print(f"✅ {name:25}: {acc:.4f} ({acc*100:.2f}%)")

BENCHMARK TRÊN TẬP VALIDATION (20% DỮ LIỆU):
✅ Bagging (Random Forest)  : 0.7878 (78.78%)
✅ Boosting (XGBoost)       : 0.8091 (80.91%)
[LightGBM] [Info] Number of positive: 3500, number of negative: 3454
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000376 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1894
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503307 -> initscore=0.013230
[LightGBM] [Info] Start training from score 0.013230
✅ Boosting (LightGBM)      : 0.8120 (81.20%)
✅ Boosting (CatBoost)      : 0.8062 (80.62%)
[LightGBM] [Info] Number of positive: 3500, number of negative: 3454
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000324 seconds.
You can set `force_row_wise=true` to remove the o

*   **Bagging (Random Forest)**: Giảm phương sai (variance) bằng cách huấn luyện nhiều cây quyết định trên các tập dữ liệu con được lấy mẫu ngẫu nhiên có hoàn lại.
*   **Boosting (XGBoost, LightGBM, CatBoost)**: Giảm độ lệch (bias) bằng cách xây dựng các mô hình tuần tự, mô hình sau tập trung sửa lỗi sai của mô hình trước.
*   **Stacking / Voting**: Kết hợp dự đoán của tất cả các mô hình trên. Trong bài này, chúng ta dùng **Soft Voting** (tính trung bình xác suất dự đoán của các mô hình) để cho ra kết quả cuối cùng ổn định nhất.

In [12]:
print("Đang huấn luyện mô hình Ensemble trên toàn bộ dữ liệu...")
voting_clf.fit(X, y)

test_predictions = voting_clf.predict(X_test_final)

submission_preds = test_predictions.astype(bool)

submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': submission_preds
})

submission.to_csv('submission.csv', index=False)
print("Đã lưu file 'submission.csv' thành công!")
submission.head()

Đang huấn luyện mô hình Ensemble trên toàn bộ dữ liệu...
[LightGBM] [Info] Number of positive: 4378, number of negative: 4315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000386 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1895
[LightGBM] [Info] Number of data points in the train set: 8693, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503624 -> initscore=0.014495
[LightGBM] [Info] Start training from score 0.014495
Đã lưu file 'submission.csv' thành công!


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
